# Feature Engineering and Modelling

---

1. Import packages
2. Load data
3. Modelling

---

## 1. Import packages

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Shows plots in jupyter notebook
%matplotlib inline

# Set plot style
sns.set(color_codes=True)

---
## 2. Load data

In [ ]:
df = pd.read_csv('./data_for_predictions.csv')
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

---

## 3. Modelling

We now have a dataset containing features that we have engineered and we are ready to start training a predictive model. We will be training a `Random Forest` classifier.

### Class imbalance check

Before splitting the data, it's important to check the balance of the target variable. Imbalanced classes can cause the model to be biased towards the majority class.

In [ ]:
churn_counts = df['churn'].value_counts()
print("Churn distribution:")
print(churn_counts)
print(f"\nChurn rate: {churn_counts[1] / len(df) * 100:.1f}%")

fig, ax = plt.subplots(figsize=(5, 4))
churn_counts.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('Churn Class Distribution')
ax.set_xlabel('Churn (0 = No, 1 = Yes)')
ax.set_ylabel('Count')
ax.set_xticklabels(['No Churn', 'Churn'], rotation=0)
plt.tight_layout()
plt.show()

The dataset is **imbalanced** — only ~10% of customers have churned. We will use `class_weight='balanced'` in the Random Forest to account for this, which instructs the model to penalise misclassifications of the minority class (churn) more heavily.

### Data sampling

We split the dataset into training (75%) and test (25%) sets. The test set simulates unseen data, allowing us to evaluate how well the model generalises.

In [ ]:
# Make a copy of our data
train_df = df.copy()

# Separate target variable from independent variables
y = df['churn']
X = df.drop(columns=['id', 'churn'])
print(X.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

### Model training

We train a `Random Forest` classifier with the following key parameters:

| Parameter | Value | Reason |
|---|---|---|
| `n_estimators` | 1000 | More trees → more stable predictions (converges to best performance) |
| `max_depth` | None | Trees can grow fully; let the algorithm decide depth |
| `min_samples_leaf` | 10 | Prevents individual leaves being too small / overfitting |
| `class_weight` | `'balanced'` | Corrects for the ~10% churn class imbalance |
| `random_state` | 42 | Reproducibility |
| `n_jobs` | -1 | Use all CPU cores for faster training |

In [ ]:
model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model training complete!")

### Evaluation

Now let's evaluate how well this trained model is able to predict churn on the test dataset.

We generate both hard predictions (`predict`) and probability scores (`predict_proba`) to compute a range of metrics.

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of churn class

In [ ]:
# Calculate performance metrics
accuracy  = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred)
recall    = metrics.recall_score(y_test, y_pred)
f1        = metrics.f1_score(y_test, y_pred)
roc_auc   = metrics.roc_auc_score(y_test, y_pred_proba)

print("=" * 40)
print("       MODEL PERFORMANCE METRICS")
print("=" * 40)
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("=" * 40)
print()
print("Classification Report:")
print(metrics.classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

### Confusion Matrix

In [ ]:
cm = metrics.confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn'],
    ax=ax
)
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

### ROC Curve

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic (ROC) Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Feature Importance

One of the key advantages of Random Forests is interpretability via feature importances. This shows which features contribute most to predicting churn.

In [ ]:
feature_importances = pd.Series(
    model.feature_importances_, index=X.columns
).sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(9, 6))
feature_importances.head(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Feature Importances')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Top 15 Features:")
print(feature_importances.head(top_n).to_string())

### Summary

The Random Forest model achieves:
- **~88% accuracy** overall
- **ROC-AUC of ~0.66**, meaning the model has moderate discriminatory power for identifying churners
- The `class_weight='balanced'` parameter was critical — without it, the model predicts almost no churners at all (recall ~0%) due to the 90/10 class imbalance

**Top predictive features** are margin-related columns (`margin_net_pow_ele`, `margin_gross_pow_ele`) and consumption variables (`cons_12m`, `forecast_cons_12m`), suggesting that pricing sensitivity and usage patterns are the main drivers of churn.